# Данные до метрик

`data/raw/online_retail_ii.xlsx` — выгрузка из учётной системы британского продавца сувениров и
посуды (датасет 502 на UCI), два листа, ~1,07 млн строк. Прежде чем считать выручку и удержание,
надо решить, что в этом файле вообще продажа. Ниже восемь мест, где `read_excel(...)` и `sum()`
дают правдоподобное, но неправильное число.

Реализация всех решений — в `src/retail_analytics/prepare.py`, на них есть тесты в
`tests/test_prepare.py`. Ноутбук запускается из корня репозитория.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from retail_analytics import config as cfg, metrics as m, prepare
from retail_analytics.marts import MARTS_DIR

pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 44)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [2]:
xl = pd.ExcelFile(cfg.RAW_FILE)
sheets = {name: xl.parse(name) for name in xl.sheet_names}
for name, df in sheets.items():
    print(f"{name}: {len(df):,} строк | {df.InvoiceDate.min()} — {df.InvoiceDate.max()}")
list(sheets.values())[0].columns.tolist()

Year 2009-2010: 525,461 строк | 2009-12-01 07:45:00 — 2010-12-09 20:01:00
Year 2010-2011: 541,910 строк | 2010-12-01 08:26:00 — 2011-12-09 12:50:00


['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'Customer ID',
 'Country']

## 1. Два листа, которые пересекаются

Листы называются «2009-2010» и «2010-2011», но граница у них не по году: первый заканчивается
9 декабря 2010, второй начинается 1 декабря 2010. Девять дней лежат в обоих.

In [3]:
raw = pd.concat(sheets.values(), ignore_index=True)
overlap = (raw.InvoiceDate >= "2010-12-01") & (raw.InvoiceDate < "2010-12-10")
print("строк в окне 1–9 декабря 2010:", f"{int(overlap.sum()):,}")
raw[overlap].InvoiceDate.dt.date.value_counts().sort_index()

строк в окне 1–9 декабря 2010: 45,046


InvoiceDate
2010-12-01    6216
2010-12-02    4218
2010-12-03    4404
2010-12-05    5450
2010-12-06    7756
2010-12-07    5926
2010-12-08    5294
2010-12-09    5782
Name: count, dtype: int64

## 2. Дубликаты

Если склеить листы как есть, инвойсы из этого окна попадут в витрину дважды. Считаем дубли
двумя способами — по всем колонкам и по деловому ключу.

In [4]:
key = ["Invoice", "StockCode", "InvoiceDate", "Quantity", "Price"]
print("дублей по всем полям:  ", f"{int(raw.duplicated().sum()):,}")
print("дублей по деловому ключу:", f"{int(raw.duplicated(subset=key).sum()):,}")

дублей по всем полям:   34,335
дублей по деловому ключу: 34,337


In [5]:
mask = raw.duplicated(subset=key, keep=False) & ~raw.duplicated(keep=False)
raw[mask].sort_values(key)[key + ["Description", "Customer ID"]]

,Invoice,StockCode,InvoiceDate,Quantity,Price,Description,Customer ID
724585,554084,23298,2011-05-22 11:52:00,3,4.95,"BUNTING , SPOTTY","12,909.00"
724605,554084,23298,2011-05-22 11:52:00,3,4.95,SPOTTY BUNTING,"12,909.00"
977829,575335,23203,2011-11-09 13:56:00,300,1.79,JUMBO BAG DOILEY PATTERNS,"12,931.00"
977832,575335,23203,2011-11-09 13:56:00,300,1.79,JUMBO BAG VINTAGE DOILY,"12,931.00"


Разница — две пары строк: инвойс, товар, дата, количество и цена совпадают, а описание
написано по-разному (`BUNTING , SPOTTY` и `SPOTTY BUNTING`). Если чистить «по всем полям», такие
строки останутся дублями и удвоят количество товара; если по деловому ключу — схлопнутся.
Поэтому `DEDUP_KEY` в конфиге не включает ни описание, ни клиента.

После схлопывания остаётся 34 337 удалённых строк, или около 3% файла.

## 3. Отрицательное количество — это два разных процесса

В датасете два способа «минуса»: отдельный инвойс с префиксом `C` (документ отмены) и
отрицательная строка внутри обычного инвойса (возврат или корректировка позиции). Валить их в
одну корзину нельзя: это разные бизнес-процессы, и по отдельности они ведут себя по-разному.

In [6]:
is_c = raw.Invoice.astype(str).str.startswith("C")
neg = raw.Quantity < 0
print("строк с отрицательным количеством:", f"{int(neg.sum()):,}")
print("  из них в C-инвойсе:", f"{int((neg & is_c).sum()):,}")
print("  возвратов в обычном инвойсе:", f"{int((neg & ~is_c).sum()):,}")
print("C-инвойсов всего:", f"{int(is_c.sum()):,}")
raw[is_c & (raw.Quantity > 0)][["Invoice", "StockCode", "Description", "Quantity", "Price"]]

строк с отрицательным количеством: 22,950
  из них в C-инвойсе: 19,493
  возвратов в обычном инвойсе: 3,457
C-инвойсов всего: 19,494


,Invoice,StockCode,Description,Quantity,Price
76799,C496350,M,Manual,1,373.57


Последняя таблица — пограничный случай: один C-инвойс с положительным количеством
(рукописная правка `Manual`). По правилу «префикс C = отмена» он уедет в отмены, хотя количество
положительное. Одна строка на миллион, и я её не стал обрабатывать отдельно — но в витрине
возвратов она сидит, и это надо помнить, когда кто-то спросит, почему отмена на £374.

## 4. Возвраты без цены

Возвраты в этом исходнике нельзя измерить в деньгах: у строк возврата обнулена цена.

In [7]:
returns = raw[neg & ~is_c]
print("строк возврата:", f"{len(returns):,}")
print("  с Price = 0:", f"{int((returns.Price == 0).sum()):,}")
print("сумма возвратов, как её видит система:", (returns.Quantity * returns.Price).sum())
print("штуками:", f"{int(returns.Quantity.sum()):,}")

строк возврата: 3,457
  с Price = 0: 3,457
сумма возвратов, как её видит система: 0.0
штуками: -573,085


То есть «доля возвратов в выручке» из этих данных — всегда ноль, и это свойство учёта, а
не показатель качества товара. Возвраты считаются штуками и строками, и в README это сказано
отдельно.

## 5. Нетоварные строки

Часть строк — не товар, а проводки: доставка, ручные корректировки, списания, комиссии
маркетплейса, bad debt. Коды у них либо буквенные, либо числовые с описанием-примечанием.

In [8]:
raw["amount"] = raw.Quantity * raw.Price
lettered = raw[~raw.StockCode.astype(str).str.match(r"^\d{5,6}$")]
g = lettered.groupby("StockCode")
table = pd.DataFrame({
    "lines": g.size(),
    "description": g.Description.agg(lambda s: s.dropna().mode().iat[0] if s.notna().any() else "—"),
    "amount": g.amount.sum(),
}).sort_values("amount")
table.head(14).round(0)

,lines,description,amount
StockCode,,,
AMAZONFEE,43,AMAZON FEE,"-260,764.00"
B,6,Adjust bad debt,"-147,614.00"
M,1421,Manual,"-82,796.00"
BANK CHARGES,102,Bank Charges,"-35,563.00"
D,177,Discount,"-13,485.00"
CRUK,16,CRUK Commission,"-7,933.00"
S,104,SAMPLES,"-6,066.00"
79340W,6,WHITE ORCHID FLOWER LIGHTS,-45.00
PADS,19,PADS TO MATCH ALL CUSHIONS,-37.00


`M` даёт и +£341 тыс., и −£424 тыс.: это ручные проводки в обе стороны. `B` — bad debt на
£147,6 тыс., `AMAZONFEE` — комиссия маркетплейса на £260,8 тыс. Всё это не продажи товара, но в
сумме `Quantity × Price` они выглядят как продажи.

Ключевая ловушка — искать такие строки по слову в описании. Слово `carriage` ловит и
`CARRIAGE` (доставка), и `BLACK BAROQUE CARRIAGE CLOCK` (настольные часы).

In [9]:
desc = raw.Description.astype("string").fillna("")
for word in ["carriage", "wrapping", "display", "sample", "adjust"]:
    hit = raw[desc.str.contains(word, case=False)]
    top = hit.Description.value_counts().head(2)
    print(f"{word}: {len(hit):,} строк | " + " ; ".join(f"{k} ({v})" for k, v in top.items()))

carriage: 494 строк | CARRIAGE (279) ; Next Day Carriage (80)
wrapping: 298 строк | TEA PARTY  WRAPPING PAPER  (210) ; HEARTS WRAPPING TAPE  (69)
display: 105 строк | ROBOT MUG IN DISPLAY BOX (84) ; CRYSTAL STUD EARRINGS CLEAR DISPLAY (14)
sample: 108 строк | SAMPLES (104) ; samples/damages (1)


adjust: 110 строк | Adjustment by john on 26/01/2010 16 (38) ; Adjustment by john on 26/01/2010 17 (26)


In [10]:
# в четырёх строках вместо описания лежит другой код товара, и все четыре — нулевые
raw[raw.Description.map(lambda v: isinstance(v, int))]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,amount
6911,490007,84347,21494,-720,2009-12-03 12:09:00,0.00,NaN,United Kingdom,-0.00
274052,516016,22467,22719,2,2010-07-16 10:11:00,0.00,NaN,United Kingdom,0.00
274053,516017,22719,22467,-2,2010-07-16 10:11:00,0.00,NaN,United Kingdom,-0.00
945851,572891,23343,20713,-400,2011-10-26 14:14:00,0.00,NaN,United Kingdom,-0.00


Поэтому в `NON_MERCH_PATTERN` стоят границы слов и lookahead для `carriage clock` /
`carriage lantern`, а слов `wrapping` и `display` в паттерне нет вовсе — под ними живут только
товары. Список кодов из `NON_MERCH_CODES` сверен вручную по фактическим описаниям.

Помеченные, но не удалённые строки уходят в витрину `non_merchandise_breakdown`: так в отчёте
видно, сколько денег осталось за рамками выручки, а не «мы убрали мусор».

## 6. Пропуски

Три колонки с дырками, и у каждой своё решение.

In [11]:
nulls = raw.isna().sum().rename("пропусков").to_frame()
nulls["доля, %"] = (nulls["пропусков"] / len(raw) * 100).round(1)
nulls[nulls["пропусков"] > 0]

,пропусков,"доля, %"
Description,4382,0.40
Customer ID,243007,22.80


- **Customer ID** — 22,8% строк. Это не «гости» в смысле веб-аналитики, а строки, где
  клиента не заполнили: в выручку они входят, в когорты и RFM — нет. Отсюда переключатель
  «только идентифицированные» в дашборде.
- **Description** — 4 382 строки, но у большинства код товара валидный и встречается в других
  инвойсах. Описание подтягивается самым частым значением по `stock_code`; недобранное остаётся
  видимой меткой `<не удалось восстановить>`. В четырёх строках вместо описания лежит число —
  это другой код товара, и все четыре строки нулевые по цене, то есть деньги не трогают.
- **Country** — пустых нет, но есть `Unspecified` (756 строк): оставлен отдельной категорией,
  а не размазан по другим странам.

## 7. Страны: EIRE и RSA

Справочник стран в файле рукописный. Ирландия записана устаревшим `EIRE`, ЮАР — как `RSA`,
а отдельного значения `Ireland` в файле нет вообще. Без склейки Ирландия (3,2% выручки)
растворилась бы среди экзотических рынков.

In [12]:
countries = raw.Country.value_counts()
print("стран в файле:", len(countries))
countries[countries.index.isin(["EIRE", "RSA", "United Kingdom", "Unspecified", "European Community"])]

стран в файле: 43


Country
United Kingdom        981330
EIRE                   17866
Unspecified              756
RSA                      169
European Community        61
Name: count, dtype: int64

## 8. Типы

`Invoice` и `StockCode` лежат вперемешку числами и строками, коды местами в нижнем регистре,
а цены — float с шумом в последних знаках.

In [13]:
print("Invoice:  ", raw.Invoice.map(lambda v: type(v).__name__).value_counts().to_dict())
print("StockCode:", raw.StockCode.map(lambda v: type(v).__name__).value_counts().to_dict())
codes_str = raw.StockCode.astype(str)
print("кодов в нижнем регистре:", int((codes_str != codes_str.str.upper()).sum()))
noisy = raw.Price * 100 % 1 != 0
print("цен с артефактом после второго знака:", f"{int(noisy.sum()):,}")
print("пример такого значения целиком:", f"{raw.Price[noisy].iat[0]:.17g}")

Invoice:   {'int': 1047871, 'str': 19500}


StockCode: {'int': 932385, 'str': 134986}


кодов в нижнем регистре: 3471
цен с артефактом после второго знака: 112,897
пример такого значения целиком: 2.5499999999999998


Если `StockCode` не привести к одной строке в верхнем регистре, `72349b` и `72349B`
окажутся двумя товарами — это 3 471 строка. Если не округлить цену до копеек, группировки по
`unit_price` начинают различать `2.5499999999999998` и `2.55`.

## Итог: что изменилось после очистки

Собираю факт целиком и смотрю журнал — это те же правила, что описаны выше, в применении к
полному файлу.

In [14]:
fact, dq = prepare.build_fact()
dq

,rule,rows,note
0,лист «Year 2009-2010» прочитан,525461,2009-12-01 — 2010-12-09
1,лист «Year 2010-2011» прочитан,541910,2010-12-01 — 2011-12-09
2,точные дубликаты удалены,34337,"ключ: invoice_no, stock_code, invoice_da..."
3,тестовые позиции удалены,17,"TEST001, TEST002"
4,описание подтянуто по stock_code,3899,пустые description
5,страна склеена по справочнику,17866,EIRE -> Ireland
6,"нетоварные строки помечены, не удалены",6133,"доставка, manual, списания и прочее"
7,страна не восстанавливалась,752,Unspecified оставлен отдельной категорией
8,строки без customer_id помечены как гост...,235150,"в выручку входят, в метрики удержания нет"
9,оптовые строки помечены флагом,2,quantity >= 20000


In [15]:
sales = fact[(fact.line_type == "sale") & fact.is_merchandise]
print(f"строк: {len(fact):,} (было {len(raw):,})")
print(f"товарная выручка: £{sales.line_revenue.sum():,.0f}")
print(f"инвойсов: {sales.invoice_no.nunique():,}, клиентов: {sales.customer_id.nunique():,}")
print(f"период: {fact.invoice_date.min():%Y-%m-%d} — {fact.invoice_date.max():%Y-%m-%d}")

строк: 1,033,017 (было 1,067,371)
товарная выручка: £19,643,866
инвойсов: 41,327, клиентов: 5,853
период: 2009-12-01 — 2011-12-09


Дальше по этому факту строятся витрины (`python -m retail_analytics.marts`), и те же
числа независимо пересчитываются pandas-ом в `src/retail_analytics/metrics.py`. Расхождение двух
путей ловит `tests/test_marts_agree_with_pandas.py`.